In [21]:
import pandas as pd
import numpy as np
import igraph as ig
import leidenalg as la
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy import sparse
import os




import celloracle as co
co.__version__


'0.10.12'

In [79]:
adata = sc.read_h5ad("MAIT_NKT_gdT_paperlabels_exact.h5ad")
adata    

AnnData object with n_obs × n_vars = 8253 × 25970
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'nGene', 'nUMI', 'use', 'final_celltype', 'major_type', 'paper_subtype', 'n_counts', 'n_genes', 'author_cluster', 'author_lineage', 'paper_cluster', 'paper_lineage'
    var: 'gene_symbols', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'gene_symbol'
    uns: 'log1p', 'paper_subtype_colors', 'umap_provenance'
    obsm: 'X_PCA', 'X_TSNE', 'X_umap'
    layers: 'counts'

In [80]:
print(f"Cell number is :{adata.shape[0]}")
print(f"Gene number is :{adata.shape[1]}")

Cell number is :8253
Gene number is :25970


In [78]:

Xc = adata.layers["counts"]
is_int_like = np.allclose(Xc.data, np.floor(Xc.data)) if sparse.issparse(Xc) else np.allclose(Xc, np.round(Xc))
print("counts integer-like:", bool(is_int_like))

map_nkt = {"N1":"NKTp","N2":"NKT1","N3":"NKT2","N4":"NKT2","N5":"NKT2","N6":"NKT2","N7":"NKT17"}
map_mait_sub = {"M1":"MAIT0","M2":"MAIT2","M3":"MAIT1/17i","M4":"MAIT1i","M5":"MAIT1","M6":"MAIT17i","M7":"MAIT17i","M8":"MAIT17"}
map_mait_stage = {"M1":"S1","M2":"S2","M3":"S2","M4":"S2","M5":"S3","M6":"S3","M7":"S3","M8":"S3"}
map_gdt = {"G1":"Tγδp","G2":"Tγδ1/17i","G3":"Tγδ1/17i","G4":"Tγδ17i","G5":"Tγδ17i","G6-1":"Tγδ17","G6-2":"Tγδ17","G7-1":"Tγδ1i","G7-2":"Tγδ1"}

def _subtype(cl):
    return map_nkt.get(cl, map_mait_sub.get(cl, map_gdt.get(cl, "NA")))
def _stage(cl):
    return map_mait_stage.get(cl, "NA")

if "paper_subtype_strict" not in adata.obs.columns:
    adata.obs["paper_subtype_strict"] = adata.obs["paper_cluster"].astype(str).map(_subtype)
if "paper_stage" not in adata.obs.columns:
    adata.obs["paper_stage"] = adata.obs["paper_cluster"].astype(str).map(_stage)

print("paper_subtype_strict top:\n", adata.obs["paper_subtype_strict"].value_counts())

print("\nCounts by type:")

print(adata.obs["paper_lineage"].value_counts())





counts integer-like: True
paper_subtype_strict top:
 NKT2         2567
Tγδ17i       1013
Tγδ1/17i      633
MAIT1/17i     562
MAIT0         532
MAIT17        464
Tγδ1i         419
MAIT17i       344
Tγδp          310
NKT17         283
Tγδ17         230
NKTp          225
NKT1          210
MAIT1i        154
MAIT1         148
MAIT2          83
Tγδ1           76
Name: paper_subtype_strict, dtype: int64

Counts by type:
NKT     3285
γδT     2681
MAIT    2287
Name: paper_lineage, dtype: int64


In [11]:
print(adata.obs["paper_cluster"].value_counts().sort_index())
print("\nCounts by strict subtype:")

N1      225
N2      210
N3      843
N4      785
N5      871
N6       68
N7      283
M1      532
M2       83
M3      562
M4      154
M5      148
M6      113
M7      231
M8      464
G1      310
G2      336
G3      297
G4      745
G5      268
G6-1    156
G6-2     74
G7-1    419
G7-2     76
Name: paper_cluster, dtype: int64

Counts by strict subtype:


In [18]:
is_NKT  = adata.obs["paper_lineage"].astype(str).str.contains("NKT", case=False, na=False)
is_MAIT = adata.obs["paper_lineage"].astype(str).str.contains("MAIT", case=False, na=False)
is_gdT  = adata.obs["paper_lineage"].astype(str).str.contains("γδ|gd|tgd|gamma", case=False, na=False, regex=True)


mask_NKTp = is_NKT & adata.obs["paper_subtype_strict"].astype(str).eq("NKTp")
mask_Tgdp = is_gdT & adata.obs["paper_subtype_strict"].astype(str).eq("Tγδp")

mask_MAIT_S2_stage = is_MAIT & adata.obs.get("paper_stage", pd.Series("", index=adata.obs.index)).astype(str).eq("S2")


clusters = adata.obs["paper_cluster"].astype(str)
mask_MAIT_S2_by_clusters = is_MAIT & clusters.isin({"M2","M3","M4"})


mask_MAIT_S2 = mask_MAIT_S2_stage | mask_MAIT_S2_by_clusters

print("NKTp cells:", int(mask_NKTp.sum()))
print("MAIT_S2 cells:", int(mask_MAIT_S2.sum()))
print("Tγδp cells:", int(mask_Tgdp.sum()))



NKTp cells: 225
MAIT_S2 cells: 799
Tγδp cells: 310


In [19]:
anchors_mask = (mask_NKTp | mask_MAIT_S2 | mask_Tgdp)
anchors = adata[anchors_mask].copy()

# HVG  on raw counts
sc.pp.highly_variable_genes(
    anchors,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=5000
)
hvg_genes_grn = anchors.var_names[anchors.var["highly_variable"]].tolist()
print("HVG for GRN:", len(hvg_genes_grn))


HVG for GRN: 5000


In [93]:
anchors

AnnData object with n_obs × n_vars = 1334 × 25970
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'nGene', 'nUMI', 'use', 'final_celltype', 'major_type', 'paper_subtype', 'n_counts', 'n_genes', 'author_cluster', 'author_lineage', 'paper_cluster', 'paper_lineage', 'paper_subtype_strict', 'paper_stage'
    var: 'gene_symbols', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'gene_symbol'
    uns: 'log1p', 'paper_subtype_colors', 'umap_provenance', 'hvg'
    obsm: 'X_PCA', 'X_TSNE', 'X_umap'
    layers: 'counts'

In [20]:

def prep_layer(mask, hvg, n_comps=50):
    A = adata[mask, hvg].copy()
    # X - raw counts
    Xc = A.layers["counts"]
    A.X = Xc.A if sparse.issparse(Xc) and hasattr(Xc, "A") else Xc.copy()

    # PCA-embedding
    B = A.copy()
    sc.pp.normalize_total(B, target_sum=1e4)
    sc.pp.log1p(B)
    sc.pp.pca(B, n_comps=n_comps)
    A.obsm["X_pca"] = B.obsm["X_pca"]

    if "gene_symbols" not in A.var.columns:
        A.var["gene_symbols"] = A.var_names
    return A

A_NKT  = prep_layer(mask_NKTp,    hvg_genes_grn)
A_MAIT = prep_layer(mask_MAIT_S2, hvg_genes_grn)
A_gdT  = prep_layer(mask_Tgdp,    hvg_genes_grn)

print("NKTp × HVG:", A_NKT.shape,  " PCA:", A_NKT.obsm["X_pca"].shape)
print("MAIT_S2 × HVG:", A_MAIT.shape, " PCA:", A_MAIT.obsm["X_pca"].shape)
print("Tγδp × HVG:", A_gdT.shape,  " PCA:", A_gdT.obsm["X_pca"].shape)


NKTp × HVG: (225, 5000)  PCA: (225, 50)
MAIT_S2 × HVG: (799, 5000)  PCA: (799, 50)
Tγδp × HVG: (310, 5000)  PCA: (310, 50)


In [22]:
A_NKT.write_h5ad("GRN_lineage_NKT.h5ad", compression="gzip")
A_MAIT.write_h5ad("GRN_lineage_MAIT_S2.h5ad", compression="gzip")
A_gdT.write_h5ad("GRN_lineage_gdT.h5ad", compression="gzip")

In [29]:
os.makedirs("outputs/grn_raw", exist_ok=True)

base_GRN = co.data.load_mouse_promoter_base_GRN(version='mm10_gimmemotifsv5_fpr2')
tfs_all = base_GRN.columns.astype(str).tolist()


Loading prebuilt promoter base-GRN. Version: mm10_gimmemotifsv5_fpr2


In [30]:
base_GRN

,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ac226150.2,Afp,Ahctf1,Ahr,Ahrr,...,Znf784,Znf8,Znf816,Znf85,Zscan10,Zscan16,Zscan22,Zscan26,Zscan31,Zscan4
0,chr10_100014821_100015921,Kitl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10_100334676_100335776,Gm4301,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr10_100339838_100340938,Gm4303,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10_100344970_100346070,Gm4302,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_100350091_100351191,Gm4302,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29303,chrY_8835069_8836169,Gm20815,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29304,chrY_896788_897888,Kdm5d,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29305,chrY_90754950_90756050,G530011O06Rik,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29306,chrY_90784442_90785542,Erdr1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [81]:
n_edges = (base_GRN.values != 0).sum()
n_tfs = base_GRN.shape[1]
n_targets = base_GRN.shape[0]
print(f"Base_GRN: {n_edges:,} edges, {n_tfs} TF, {n_targets} targets")

Base_GRN: 4,159,372 edges, 1096 TF, 29308 targets


In [31]:
tfs_all

['peak_id',
 'gene_short_name',
 '9430076c15rik',
 'Ac002126.6',
 'Ac012531.1',
 'Ac226150.2',
 'Afp',
 'Ahctf1',
 'Ahr',
 'Ahrr',
 'Aire',
 'Al592170.2',
 'Al662824.5',
 'Al662828.6',
 'Al662830.5',
 'Al662833.4',
 'Al662834.13',
 'Al773544.1',
 'Al844527.7',
 'Al844853.8',
 'Al845464.3',
 'Alx1',
 'Alx3',
 'Alx4',
 'Amyb',
 'Ap1',
 'Ap2alpha',
 'Ap3',
 'Ar',
 'Arhgef12',
 'Arid2',
 'Arid3a',
 'Arid3b',
 'Arid3c',
 'Arid5a',
 'Arid5b',
 'Arnt',
 'Arnt2',
 'Arntl',
 'Arntl2',
 'Arx',
 'Ascl1',
 'Ascl2',
 'Atf1',
 'Atf2',
 'Atf3',
 'Atf4',
 'Atf5',
 'Atf6',
 'Atf6b',
 'Atf7',
 'Atoh1',
 'Atoh7',
 'Atoh8',
 'Bach1',
 'Bach2',
 'Barhl1',
 'Barhl2',
 'Barx1',
 'Barx2',
 'Batf',
 'Batf3',
 'Bbx',
 'Bcl11a',
 'Bcl11b',
 'Bcl3',
 'Bcl6',
 'Bcl6b',
 'Bclaf1',
 'Bdp1',
 'Bhlha15',
 'Bhlhb2',
 'Bhlhb3',
 'Bhlhe22',
 'Bhlhe23',
 'Bhlhe40',
 'Bhlhe41',
 'Bmal1',
 'Bmyb',
 'Bptf',
 'Brachyury',
 'Brca1',
 'Brf1',
 'Brf2',
 'Brn1',
 'Brn2',
 'Bsx',
 'Bx088580.2',
 'Bx284686.1',
 'Bx927239.6',
 'C11o

In [32]:
def add_oracle_cluster(h5_path, label, color="#1f77b4"):
    A = sc.read_h5ad(h5_path)
    A.obs["oracle_cluster"] = label
    A.uns["oracle_cluster_colors"] = [color]
    A.write_h5ad(h5_path, compression="gzip")
    print(f"Patched {h5_path}: oracle_cluster='{label}', colors set.")

add_oracle_cluster("GRN_lineage_NKT.h5ad",      "NKTp",   "#1f77b4")
add_oracle_cluster("GRN_lineage_MAIT_S2.h5ad",  "MAIT_S2","#2ca02c")
add_oracle_cluster("GRN_lineage_gdT.h5ad",      "Tγδp",   "#d62728")


Patched GRN_lineage_NKT.h5ad: oracle_cluster='NKTp', colors set.
Patched GRN_lineage_MAIT_S2.h5ad: oracle_cluster='MAIT_S2', colors set.
Patched GRN_lineage_gdT.h5ad: oracle_cluster='Tγδp', colors set.


In [33]:
def run_oracle_links(h5ad_path, alpha=10, n_pca=50, k_frac=0.025):
    A = sc.read_h5ad(h5ad_path)
    k = max(10, int(k_frac * A.n_obs))
    ora = co.Oracle()
    ora.import_anndata_as_raw_count(
        adata=A,
        cluster_column_name="oracle_cluster", 
        embedding_name="X_pca"
    )
    ora.import_TF_data(TF_info_matrix=base_GRN.loc[:, [t for t in tfs_all if t in base_GRN.columns]])
    ora.perform_PCA(n_components=n_pca)
    ora.knn_imputation(k=k, balanced=True, b_sight=150, b_maxl=150)
    links = ora.get_links(alpha=alpha, n_jobs=-1)
    return links

links_NKT  = run_oracle_links("GRN_lineage_NKT.h5ad")
links_MAIT = run_oracle_links("GRN_lineage_MAIT_S2.h5ad")
links_gdT  = run_oracle_links("GRN_lineage_gdT.h5ad")



5000 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.


  0%|          | 0/1 [00:00<?, ?it/s]

5000 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.


  0%|          | 0/1 [00:00<?, ?it/s]

5000 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.


  0%|          | 0/1 [00:00<?, ?it/s]

In [37]:

def links_to_df(links_obj, preferred_label=None):

    if hasattr(links_obj, "links_dict") and isinstance(links_obj.links_dict, dict) and links_obj.links_dict:
        labels = list(links_obj.links_dict.keys())
        label = preferred_label if (preferred_label in labels) else labels[0]
        df = links_obj.links_dict[label].copy()
    elif hasattr(links_obj, "links"):
        df = pd.DataFrame(links_obj.links).copy()
    else:
        raise RuntimeError("No links_dict")

    if "coef_abs" not in df.columns:
        if "coef" in df.columns:
            df["coef_abs"] = df["coef"].abs()
        elif "weight" in df.columns:
            df["coef_abs"] = df["weight"].abs()
        else:
            raise ValueError("No coef, weight ")


    if "source" not in df.columns or "target" not in df.columns:

        if "tf" in df.columns and "target" in df.columns:
            df = df.rename(columns={"tf":"source"})
        else:
            raise ValueError("No source/target")

    return df[["source","target","coef_abs"]].rename(columns={"coef_abs":"weight"})




In [38]:
df_NKT  = links_to_df(links_NKT,  preferred_label="NKTp")
df_MAIT = links_to_df(links_MAIT, preferred_label="MAIT_S2")
df_gdT  = links_to_df(links_gdT,  preferred_label="Tγδp")

for name, df in {"NKT":df_NKT, "MAIT":df_MAIT, "gdT":df_gdT}.items():
    print(name, df.shape, df.head(3))


NKT (156144, 3)   source         target  weight
0   Smc3  0610009E02Rik     0.0
1   Ctcf  0610009E02Rik     0.0
2    Sp1  0610009E02Rik     0.0
MAIT (156144, 3)   source         target    weight
0   Smc3  0610009E02Rik  0.000960
1   Ctcf  0610009E02Rik  0.000670
2    Sp1  0610009E02Rik  0.001026
gdT (156144, 3)   source         target    weight
0   Smc3  0610009E02Rik  0.000897
1   Ctcf  0610009E02Rik  0.001561
2    Sp1  0610009E02Rik  0.001219


In [39]:

os.makedirs("outputs/grn_raw", exist_ok=True)

df_NKT.to_csv("outputs/grn_raw/raw_GRN_for_NKTp.tsv", sep="\t", index=False)
df_MAIT.to_csv("outputs/grn_raw/raw_GRN_for_MAIT_S2.tsv", sep="\t", index=False)
df_gdT.to_csv("outputs/grn_raw/raw_GRN_for_Tgdp.tsv",   sep="\t", index=False)

print("Saved:",
      "outputs/grn_raw/raw_GRN_for_NKTp.tsv",
      "outputs/grn_raw/raw_GRN_for_MAIT_S2.tsv",
      "outputs/grn_raw/raw_GRN_for_Tgdp.tsv", sep="\n - ")


Saved:
 - outputs/grn_raw/raw_GRN_for_NKTp.tsv
 - outputs/grn_raw/raw_GRN_for_MAIT_S2.tsv
 - outputs/grn_raw/raw_GRN_for_Tgdp.tsv


In [42]:
for name, df in {"NKT": df_NKT, "MAIT": df_MAIT, "gdT": df_gdT}.items():
    print(f"\n{name}  shape {df.shape}")
    print("Columns:", df.columns.tolist())
    print(df.head())



NKT  shape (156144, 3)
Columns: ['source', 'target', 'weight']
  source         target  weight
0   Smc3  0610009E02Rik     0.0
1   Ctcf  0610009E02Rik     0.0
2    Sp1  0610009E02Rik     0.0
3  Stat1  0610009E02Rik     0.0
4   Klf9  0610009E02Rik     0.0

MAIT  shape (156144, 3)
Columns: ['source', 'target', 'weight']
  source         target    weight
0   Smc3  0610009E02Rik  0.000960
1   Ctcf  0610009E02Rik  0.000670
2    Sp1  0610009E02Rik  0.001026
3  Stat1  0610009E02Rik  0.000275
4   Klf9  0610009E02Rik  0.000268

gdT  shape (156144, 3)
Columns: ['source', 'target', 'weight']
  source         target    weight
0   Smc3  0610009E02Rik  0.000897
1   Ctcf  0610009E02Rik  0.001561
2    Sp1  0610009E02Rik  0.001219
3  Stat1  0610009E02Rik  0.000518
4   Klf9  0610009E02Rik  0.000726


In [43]:
def peek_links_full(links_obj, prefer_label=None, n=5):

    keys = list(getattr(links_obj, "links_dict", {}).keys())
    print("clusters in links_dict:", keys)
    lab = prefer_label if (prefer_label in keys) else (keys[0] if keys else None)

    df_full = links_obj.links_dict[lab]
    print(f"\n[{lab}] shape:", df_full.shape)
    print("columns:", df_full.columns.tolist())
    display(df_full.head(n))
    return lab, df_full

lab_nkt,  nkt_full  = peek_links_full(links_NKT,  prefer_label="NKTp")
lab_mait, mait_full = peek_links_full(links_MAIT, prefer_label="MAIT_S2")
lab_gdt,  gdt_full  = peek_links_full(links_gdT,  prefer_label="Tγδp")


clusters in links_dict: ['NKTp']

[NKTp] shape: (156144, 6)
columns: ['source', 'target', 'coef_mean', 'coef_abs', 'p', '-logp']


,source,target,coef_mean,coef_abs,p,-logp
0,Smc3,0610009E02Rik,0.0,0.0,NaN,-0.0
1,Ctcf,0610009E02Rik,0.0,0.0,NaN,-0.0
2,Sp1,0610009E02Rik,0.0,0.0,NaN,-0.0
3,Stat1,0610009E02Rik,0.0,0.0,NaN,-0.0
4,Klf9,0610009E02Rik,0.0,0.0,NaN,-0.0


clusters in links_dict: ['MAIT_S2']

[MAIT_S2] shape: (156144, 6)
columns: ['source', 'target', 'coef_mean', 'coef_abs', 'p', '-logp']


,source,target,coef_mean,coef_abs,p,-logp
0,Smc3,0610009E02Rik,-0.000960,0.000960,1.220642e-07,6.913412
1,Ctcf,0610009E02Rik,-0.000670,0.000670,2.202796e-05,4.657026
2,Sp1,0610009E02Rik,0.001026,0.001026,4.061639e-09,8.391299
3,Stat1,0610009E02Rik,-0.000275,0.000275,2.016288e-02,1.695447
4,Klf9,0610009E02Rik,-0.000268,0.000268,3.609747e-03,2.442523


clusters in links_dict: ['Tγδp']

[Tγδp] shape: (156144, 6)
columns: ['source', 'target', 'coef_mean', 'coef_abs', 'p', '-logp']


,source,target,coef_mean,coef_abs,p,-logp
0,Smc3,0610009E02Rik,-0.000897,0.000897,0.006959,2.157443
1,Ctcf,0610009E02Rik,0.001561,0.001561,0.038581,1.413627
2,Sp1,0610009E02Rik,-0.001219,0.001219,0.000344,3.463246
3,Stat1,0610009E02Rik,0.000518,0.000518,0.039273,1.405907
4,Klf9,0610009E02Rik,0.000726,0.000726,0.000130,3.887431


In [48]:
# filtretion GRN by FDR rom p-val

from statsmodels.stats.multitest import multipletests
os.makedirs("outputs/grn", exist_ok=True)

def filter_by_fdr_from_links(links_obj, label, out_path, fdr_thr=0.05, min_weight=None, sort_and_keep=None):

    lab_keys = list(getattr(links_obj, "links_dict", {}).keys())
    df = links_obj.links_dict[label].copy()

    if "coef_abs" in df.columns:
        df["weight"] = df["coef_abs"].astype(float)
    elif "coef" in df.columns:
        df["weight"] = df["coef"].astype(float).abs()
    else:
        raise ValueError("Не нашёл ни 'coef_abs', ни 'coef' для расчёта веса.")

    # p - FDR 
    if "p" not in df.columns:
        raise ValueError("В Links нет p-value ('p'). Нужен запуск get_links с расчётом p.")
    p = pd.to_numeric(df["p"], errors="coerce").fillna(1.0).values
    q = multipletests(p, method="fdr_bh")[1]
    df["fdr"] = q

   
    kept = df[df["fdr"] <= fdr_thr].copy()
    if min_weight is not None:
        kept = kept[kept["weight"] >= float(min_weight)].copy()


    if sort_and_keep is not None:
        kept = kept.sort_values("weight", ascending=False).head(int(sort_and_keep)).copy()


    cols_to_save = ["source","target","weight","p","fdr"]
    kept[cols_to_save].to_csv(out_path, sep="\t", index=False)

    n_total = df.shape[0]
    n_valid_p = np.isfinite(p).sum()
    n_kept = kept.shape[0]
    n_tfs = kept["source"].nunique()
    n_tg  = kept["target"].nunique()

    print(f"[{label}] total={n_total}, valid_p={n_valid_p}, kept(FDR≤{fdr_thr})={n_kept}, "
          f"unique_TF={n_tfs}, unique_targets={n_tg} → {out_path}")
    return kept


fdr_thr = 0.05       
min_w   = None       
cap_N   = None       

grn_nkt  = filter_by_fdr_from_links(links_NKT,  "NKTp",    "outputs/grn/grn_NKT_edges.csv",  fdr_thr=fdr_thr, min_weight=min_w, sort_and_keep=cap_N)
grn_mait = filter_by_fdr_from_links(links_MAIT, "MAIT_S2", "outputs/grn/grn_MAIT_edges.csv", fdr_thr=fdr_thr, min_weight=min_w, sort_and_keep=cap_N)
grn_gdt  = filter_by_fdr_from_links(links_gdT,  "Tγδp",    "outputs/grn/grn_gdT_edges.csv",  fdr_thr=fdr_thr, min_weight=min_w, sort_and_keep=cap_N)


summary = pd.DataFrame({
    "layer": ["NKTp","MAIT_S2","Tγδp"],
    "edges_kept": [grn_nkt.shape[0], grn_mait.shape[0], grn_gdt.shape[0]],
    "TFs": [grn_nkt["source"].nunique(), grn_mait["source"].nunique(), grn_gdt["source"].nunique()],
    "targets": [grn_nkt["target"].nunique(), grn_mait["target"].nunique(), grn_gdt["target"].nunique()],
})




[NKTp] total=156144, valid_p=156144, kept(FDR≤0.05)=101847, unique_TF=150, unique_targets=3648 → outputs/grn/grn_NKT_edges.csv
[MAIT_S2] total=156144, valid_p=156144, kept(FDR≤0.05)=93881, unique_TF=143, unique_targets=3407 → outputs/grn/grn_MAIT_edges.csv
[Tγδp] total=156144, valid_p=156144, kept(FDR≤0.05)=109224, unique_TF=158, unique_targets=3719 → outputs/grn/grn_gdT_edges.csv


In [49]:
summary

,layer,edges_kept,TFs,targets
0,NKTp,101847,150,3648
1,MAIT_S2,93881,143,3407
2,Tγδp,109224,158,3719


In [60]:
df_nkt  = pd.read_csv("outputs/grn/grn_NKTp.tsv", sep="\t")
df_mait = pd.read_csv("outputs/grn/grn_MAIT_S2.tsv", sep="\t")
df_gdt  = pd.read_csv("outputs/grn/grn_Tgdp.tsv", sep="\t")

print("NKTp:", df_nkt.shape)
print("MAIT_S2:", df_mait.shape)
print("Tγδp:", df_gdt.shape)


NKTp: (20000, 4)
MAIT_S2: (20000, 4)
Tγδp: (20000, 4)


In [62]:
df_mait

,source,target,weight,p
0,Foxo1,Ms4a4b,0.403499,4.237377e-20
1,Klf2,Ccr9,0.369106,1.886060e-15
2,Klf2,Ifi27l2a,0.348540,1.098010e-12
3,Klf2,Rps8,0.325444,1.813481e-19
4,Foxo1,Rpl36a,0.286423,1.508586e-17
...,...,...,...,...
19995,Zbtb7b,Sdf2l1,0.006570,2.161239e-08
19996,Eomes,Ifi44,0.006570,2.305631e-10
19997,Trim28,Mtg1,0.006570,4.515226e-09
19998,Batf,Mical1,0.006569,5.606407e-07


In [63]:
def tf_targets_map(df):
    m = {}
    for s, t in zip(df["source"], df["target"]):
        m.setdefault(s, set()).add(t)
    return m

TBT = {
    "NKT":  tf_targets_map(df_nkt),
    "MAIT": tf_targets_map(df_mait),
    "gdt":  tf_targets_map(df_gdt),
}

TF_all = sorted(set(TBT["NKT"]) | set(TBT["MAIT"]) | set(TBT["gdt"]))
print("TF:", len(TF_all))


TF: 162


In [64]:
from itertools import combinations

def tftf_edges_from_jaccard(tf_list, targets_map):
    rows = []
    for a, b in combinations(tf_list, 2):
        ta, tb = targets_map.get(a, set()), targets_map.get(b, set())
        if not ta or not tb:
            continue
        u = len(ta | tb)
        if u == 0: 
            continue
        w = len(ta & tb) / u
        if w > 0:
            rows.append((a, b, w))
    return pd.DataFrame(rows, columns=["src","tgt","weight"])

E_nkt  = tftf_edges_from_jaccard(TF_all, TBT["NKT"])
E_mait = tftf_edges_from_jaccard(TF_all, TBT["MAIT"])
E_gdt  = tftf_edges_from_jaccard(TF_all, TBT["gdt"])

print("TF–TF edges:")
print("NKT:",  E_nkt.shape)
print("MAIT:", E_mait.shape)
print("gdt:",  E_gdt.shape)


TF–TF edges:
NKT: (7974, 3)
MAIT: (8484, 3)
gdt: (8413, 3)


In [65]:
import leidenalg as la

TF_DIR_MPX = "outputs/TF_Graphs_multiplex_new"
os.makedirs(TF_DIR_MPX, exist_ok=True)

percent_steps = list(range(100, 0, -5)) 
summary = []

for pct in percent_steps:
    nkt_p  = threshold_df(E_nkt,  pct)
    mait_p = threshold_df(E_mait, pct)
    gdt_p  = threshold_df(E_gdt,  pct)

    G_nkt  = build_layer_graph(nkt_p,  tf_list, undirected=True)
    G_mait = build_layer_graph(mait_p, tf_list, undirected=True)
    G_gdt  = build_layer_graph(gdt_p,  tf_list, undirected=True)

    parts = [
        la.RBConfigurationVertexPartition(G_nkt,  weights='weight', resolution_parameter=1.0),
        la.RBConfigurationVertexPartition(G_mait, weights='weight', resolution_parameter=1.0),
        la.RBConfigurationVertexPartition(G_gdt,  weights='weight', resolution_parameter=1.0),
    ]


    Ecounts = np.array([G_nkt.ecount(), G_mait.ecount(), G_gdt.ecount()], dtype=float)
    layer_weights = (Ecounts.mean() / np.maximum(Ecounts, 1)).tolist()

    opt = la.Optimiser()
    _delta = opt.optimise_partition_multiplex(parts, layer_weights=layer_weights, n_iterations=-1)

    comm = np.array(parts[0].membership, dtype=int)

    out_comm = os.path.join(TF_DIR_MPX, f"tf_multiplex_communities_step_{pct}pct.csv")
    pd.DataFrame({"TF": tf_list, "community": comm}).to_csv(out_comm, index=False)


    step_dir = os.path.join(TF_DIR_MPX, f"edges_step_{pct}pct")
    os.makedirs(step_dir, exist_ok=True)
    nkt_p.to_csv(os.path.join(step_dir, "NKT_edges.csv"),  index=False)
    mait_p.to_csv(os.path.join(step_dir, "MAIT_edges.csv"), index=False)
    gdt_p.to_csv(os.path.join(step_dir, "gdt_edges.csv"),  index=False)

    summary.append({
        "step_pct": pct,
        "n_nodes": len(tf_list),
        "n_edges_NKT":  int(G_nkt.ecount()),
        "n_edges_MAIT": int(G_mait.ecount()),
        "n_edges_gdt":  int(G_gdt.ecount()),
        "n_communities": int(len(set(comm))),
        "singletons": int(pd.Series(comm).value_counts().eq(1).sum())
    })
    print(f"{pct}% -> comm={len(set(comm))}, singletons={summary[-1]['singletons']}, "
          f"|E| NKT/MAIT/gdt = {G_nkt.ecount()}/{G_mait.ecount()}/{G_gdt.ecount()}")

pd.DataFrame(summary).sort_values("step_pct", ascending=False).to_csv(
    os.path.join(TF_DIR_MPX, "tf_multiplex_summary.txt"), sep="\t", index=False
)


100% -> comm=4, singletons=1, |E| NKT/MAIT/gdt = 7974/8484/8413
95% -> comm=69, singletons=48, |E| NKT/MAIT/gdt = 399/425/421
90% -> comm=36, singletons=22, |E| NKT/MAIT/gdt = 798/849/842
85% -> comm=12, singletons=5, |E| NKT/MAIT/gdt = 1197/1273/1268
80% -> comm=9, singletons=3, |E| NKT/MAIT/gdt = 1603/1698/1684
75% -> comm=8, singletons=3, |E| NKT/MAIT/gdt = 1996/2121/2104
70% -> comm=8, singletons=3, |E| NKT/MAIT/gdt = 2392/2545/2524
65% -> comm=7, singletons=3, |E| NKT/MAIT/gdt = 2792/2971/2958
60% -> comm=6, singletons=2, |E| NKT/MAIT/gdt = 3192/3395/3394
55% -> comm=6, singletons=2, |E| NKT/MAIT/gdt = 3589/3822/3805
50% -> comm=6, singletons=2, |E| NKT/MAIT/gdt = 3995/4244/4214
45% -> comm=6, singletons=2, |E| NKT/MAIT/gdt = 4388/4666/4652
40% -> comm=6, singletons=2, |E| NKT/MAIT/gdt = 4789/5096/5049
35% -> comm=5, singletons=1, |E| NKT/MAIT/gdt = 5184/5539/5475
30% -> comm=4, singletons=1, |E| NKT/MAIT/gdt = 5582/5942/5889
25% -> comm=4, singletons=1, |E| NKT/MAIT/gdt = 5980/63

In [7]:
comm80 = pd.read_csv("outputs/TF_Graphs_multiplex_new/tf_multiplex_communities_step_80pct.csv")
comm80 = comm80.sort_values(["community","TF"]).reset_index(drop=True)
comm80.to_csv("outputs/TF_Graphs_multiplex_new/tf_multiplex_communities_step_80pct.csv", index=False)


tf_list = comm80["TF"].tolist()
pd.Series(tf_list, name="TF").to_csv("outputs/TF_Graphs_multiplex_new/tf_list_new.txt", index=False, header=False)

print("n_TF:", len(tf_list))
comm80.head()


n_TF: 163


,TF,community
0,Afp,0
1,Arid3a,0
2,Arid5b,0
3,Atf5,0
4,Bcl11b,0


In [8]:
adata_ctrl_hvg = sc.read_h5ad("ctrl_hvg_allphases.h5ad")  


A = adata_ctrl_hvg.copy()
X = A.layers["counts"].A if hasattr(A.layers["counts"], "A") else A.layers["counts"]
X = X / (X.sum(axis=1, keepdims=True) + 1e-8) * 1e4
X = np.log1p(X)

X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, ddof=1, keepdims=True) + 1e-8)


n_comm = comm80["community"].nunique()
pc1_mat = np.zeros((A.n_obs, n_comm), dtype=float)

for k, grp in comm80.groupby("community"):
    genes = [g for g in grp["TF"].tolist() if g in A.var_names]
    if len(genes) == 0:
        continue
    idx = [A.var_names.get_loc(g) for g in genes]
    Xg = X[:, idx]               

    U, S, Vt = np.linalg.svd(Xg, full_matrices=False)
    pc1 = U[:, 0] * S[0]        
    sign = np.sign(np.corrcoef(pc1, Xg.mean(axis=1))[0,1])
    pc1_mat[:, k] = pc1 * (sign if sign != 0 else 1.0)

pc1_df = pd.DataFrame(pc1_mat, index=A.obs_names, columns=[f"comm_{i}" for i in range(n_comm)])
pc1_df.to_csv("outputs/TF_Graphs_multiplex_new/community_pc1_control.csv")
print(pc1_df.shape, "saved outputs/TF_Graphs_multiplex_new/community_pc1_control.csv")
pc1_df.iloc[:5, :5]


(6442, 9) saved outputs/TF_Graphs_multiplex_new/community_pc1_control.csv


,comm_0,comm_1,comm_2,comm_3,comm_4
F3-AAACCTGAGAGGTTAT,-1.890984,-1.047647,-0.156059,0.089509,-0.275854
F3-AAACCTGCAATCGAAA,-1.429207,0.122839,-1.466461,-0.415547,-0.639182
F3-AAACGGGAGAGTCGGT,-2.238503,-1.454389,-1.143905,-1.258728,-0.280548
F3-TACCTTATCGACAGCC,-1.595520,-1.343174,-1.100345,-0.923606,-0.275087
F3-TACGGGCAGTGGGCTA,-1.230536,-0.701207,-2.204339,-0.807140,-0.680293


In [9]:
grn_NKT  = pd.read_csv("outputs/grn/grn_NKT_edges.csv",  sep="\t") 
grn_MAIT = pd.read_csv("outputs/grn/grn_MAIT_edges.csv", sep="\t")
grn_gdT  = pd.read_csv("outputs/grn/grn_gdT_edges.csv",  sep="\t")


GRN_all = pd.concat([grn_NKT, grn_MAIT, grn_gdT], ignore_index=True)
GRN_all = GRN_all[GRN_all["fdr"] <= 0.05]

tf_set      = set(comm80["TF"])
targets_set = set(GRN_all["target"])
pool_all    = tf_set | targets_set

print("TF:", len(tf_set), "targets:", len(targets_set), "pool_all:", len(pool_all))



TF: 163 targets: 3911 pool_all: 3911


In [11]:
hvg_5000 = set(adata_ctrl_hvg.var_names.tolist())

feat = sorted(pool_all & hvg_5000)

missing_tf = [t for t in tf_set if t not in feat and t in adata_ctrl_hvg.var_names]
feat = feat + missing_tf

print("Final features:", len(feat), "| missing TF added:", len(missing_tf))



Final features: 2685 | missing TF added: 0
